# Constructing Circuit Representation

The **circuit representation** is the form closest to how a continuous-variable circuit is usually drawn:
modes, and the operations applied to them in order. It is the one to write unless you have a reason not
to. The simulator runs it as it is, and a client compiles it to the machine's own representation for you.

This page covers all of it: creating a circuit, placing a state on a mode, applying gates, measuring,
feedforward, composing two circuits, and saving one to a file. The [Quickstart](quickstart.ipynb) is the
five-minute version.

## Creating an empty circuit

To construct a quantum circuit in the circuit representation, you first need to create an instance of the {py:class}`~claveles.circuit.CircuitRepr` class.
You can create an instance `c` with the name "empty" as shown below:

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

In [ ]:
from claveles.circuit import CircuitRepr

c = CircuitRepr("empty")

The assigned name can be retrieved using the {py:attr}`~claveles.circuit.CircuitRepr.name` property:

In [ ]:
print(c.name)

(sec:circuit-placing-states)=

## Placing a state on a mode

A circuit starts out with no modes. A mode comes into existence when you first name it with
{py:meth}`~claveles.circuit.CircuitRepr.Q`, and the first thing you do to it settles where its input comes
from. So the natural first operation on a mode is to place a state on it, written with `|` just like a
gate:

In [ ]:
from claveles.circuit import CircuitRepr
from claveles.circuit.state import QuantumState

c = CircuitRepr("placing_states")
c.Q(0) | QuantumState.squeezed(r=0.5)
c.Q(1) | QuantumState.vacuum()

print("#modes      =", c.n_modes)
print("#operations =", c.n_operations)

A placement is an operation in the sequence like any other, which is why `#operations` counts two
above. It has to come first, though: once a gate or a measurement has touched a mode, that mode's
input is already decided and placing a state on it is an error.

A circuit uses exactly the modes something has been applied to. Naming one says nothing about the
modes below it, so the indices need not be contiguous:

In [ ]:
c = CircuitRepr("sparse_modes")
c.Q(0) | QuantumState.vacuum()
c.Q(9) | QuantumState.squeezed(r=0.5)

print("modes   =", c.modes)
print("#modes  =", c.n_modes)

{py:attr}`~claveles.circuit.CircuitRepr.modes` lists them and
{py:attr}`~claveles.circuit.CircuitRepr.n_modes` counts them, so a circuit using modes 0 and 9 has two
modes rather than ten.

### Modes you leave open

A mode nothing places a state on is left **open**: the circuit says what to do with the mode but not
where its input comes from, leaving that to whatever runs the circuit.

What you may place, and what happens to an open mode, depends on the client the circuit is run with
(see {ref}`Clients <sec:clients>`):

| | {py:class}`~claveles.client.optqc_client.OptQCClient` (hardware) | {py:class}`~claveles.client.simulator_client.SimulatorClient` (simulation) |
| --- | --- | --- |
| **State used as placed** | {py:meth}`~claveles.circuit.state.QuantumState.hardware_squeezed` only (the {ref}`squeezing angle <sec:squeezing-level-and-squeezing-angle>` is the sole configurable parameter) | Any {py:class}`~claveles.circuit.state.QuantumState`, whose components may be bosonic, Gaussian or Fock |
| **Mode left open** | Runs as the machine's own squeezed state | Raises a `ValueError` &mdash; the simulator has no squeezing level to invent |
| **Unsupported state on a mode** | Raises a `ValueError` on conversion; leave the mode open instead | Raises a `ValueError` |
| **Mixing representations across modes** | &mdash; | Allowed: the bosonic states are converted into the Fock basis, reported by a {py:class}`~claveles.warnings.StateConversionWarning`. `state_representation="bosonic"` refuses such a circuit instead |

In short: for {py:class}`~claveles.client.simulator_client.SimulatorClient`, place a state on **every**
mode. For {py:class}`~claveles.client.optqc_client.OptQCClient` leaving modes open is not just allowed but
the usual thing to do, since the machine supplies its own squeezed states anyway.

For what the states themselves are, see {ref}`States <sec:state-primitives>`, and for building them
{ref}`Working with Quantum States <sec:quantum-states-in-the-sdk>`. The {ref}`Support Matrix <sec:support>` lists the same answers
for every state and operation, and is generated from the table the SDK itself consults.

```{seealso}
See {ref}`this section<sec:execution-resource-and-initialized-states>` (for {py:class}`~claveles.client.optqc_client.OptQCClient` execution) and {ref}`this section<sec:simulation-resource-and-initialized-states>` (for {py:class}`~claveles.client.simulator_client.SimulatorClient` execution) for the squeezing parameters each backend uses.
```

(sec:circuit-drawing)=

## Drawing a circuit

A circuit can be drawn, which is the quickest way to check that it holds what you meant it to. The
figure is derived from the circuit object itself, so it shows what the code above it actually built
rather than what it was meant to build, and the rest of this page uses it to show each example:


In [ ]:
from claveles.circuit.visualize import make_figure

drawn = CircuitRepr("drawn")
drawn.Q(0) | QuantumState.squeezed(r=0.5)
drawn.Q(1) | QuantumState.vacuum()

make_figure(drawn);

Each mode is a horizontal line, and each operation a box on the modes it applies to, drawn in the
order the operations were written. What to read off it:

* A **state placed** on a mode is the $|\psi\rangle$ box at the left end of its line. Every state is
  drawn with that one symbol, so which state it is has to be read off the code; what the figure tells
  you is *which modes carry one*. A mode with no box is one you left {ref}`open <sec:circuit-arity>`.
* A **gate** carries the symbol {ref}`Gates & Operations <sec:gates-symbols>` gives it, with its parameter values
  written above the box. A gate on two modes is one box spanning both.
* A parameter that comes from a **measurement** has no value yet, so it is drawn as an arrow from the
  measurement supplying it, labelled with the parameter it drives — see
  {ref}`sec:circuit-adding-feedforward`.

{py:func}`~claveles.circuit.visualize.make_figure` returns a {py:class}`matplotlib.figure.Figure`, which
Jupyter displays and {py:func}`matplotlib.pyplot.show` shows;
{py:func}`~claveles.circuit.visualize.savefig` writes one straight to a file. Both need `matplotlib`.
[Visualizing Circuit Representation](viz_circuit_repr.ipynb) covers the options they take, including
`show_states=False` to leave the state boxes out.


## Adding gates

The gates you will normally use are defined in the {py:mod}`claveles.circuit.ops.std` module, which provides the operations that are common in quantum computing. Every gate in this guide comes from that module.

Underneath, the Gaussian operations are expressed in terms of the {py:mod}`~claveles.circuit.ops.intrinsic` module, whose gates are the low-level program elements that the quad-rail lattice (QRL) CV-QC hardware implements natively. A standard Gaussian gate is normally expanded into intrinsic gates on its way to being run, and the {ref}`standard-gate reference <sec:gates-std>` gives the expansion of each one, noting the gate that reaches the simulator unexpanded. The gates the machine cannot run at all, such as {py:class}`~claveles.circuit.ops.std.Kerr`, have no intrinsic form to expand into and run on the simulator only -- {ref}`Support Matrix <sec:support>` lists where each gate runs. You only need the intrinsic gates when you work with the graph or machinery representation directly; they are listed for reference in {ref}`Gates & Operations <sec:gates-intrinsic>`.

You can add gates to a circuit by specifying the modes to which they apply.
For instance, you can apply the {py:class}`~claveles.circuit.ops.std.PhaseRotation` with $\phi = \pi$ to mode `0` of `c` as shown below:


In [ ]:
from math import pi

from claveles.circuit.ops.std import PhaseRotation

c = CircuitRepr("single_mode_gate")
c.Q(0) | PhaseRotation(phi=pi)

make_figure(c);

You can apply multiple gates to the same mode by chaining.
For example,

* `c.Q(0) | PhaseRotation(phi=pi / 2)` and `c.Q(0) | Displacement(1 + 1j)` apply two gates separately.
* `c.Q(0) | PhaseRotation(phi=pi / 2) | Displacement(1 + 1j)` chains them together in a single statement.

{py:class}`~claveles.circuit.ops.std.Displacement` takes the complex amplitude $\alpha$, which is what the
figures on this page show it with. The shift it produces in phase space is
$\sqrt{2}\,(\mathrm{Re}\,\alpha, \mathrm{Im}\,\alpha)$, so $\alpha = 1 + i$ moves the state by
$\sqrt{2}$ along each axis; {ref}`sec:gate-displacement` gives the relation both ways round.


In [ ]:
from claveles.circuit.ops.std import Displacement

c = CircuitRepr("chain_of_single_mode_gates")
c.Q(0) | PhaseRotation(phi=pi / 2) | Displacement(1 + 1j)

make_figure(c);

For two-mode gates, you need to specify both modes (in this case, mode `0` and mode `1`):


In [ ]:
from claveles.circuit.ops.std import ControlledZ

c = CircuitRepr("two_mode_gate")
c.Q(0, 1) | ControlledZ(g=1)

make_figure(c);

For each mode, gates are applied in the order they are written.

A measurement consumes the mode it reads, so applying anything to a measured mode raises a
`RuntimeError`; see {ref}`sec:circuit-measuring`.

The following example demonstrates a circuit with multiple gates:


In [ ]:
from math import pi

from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import ControlledZ, Displacement, MeasureHomodyne, PhaseRotation

c = CircuitRepr("multiple_gates")
c.Q(0) | PhaseRotation(phi=pi / 4) | Displacement(1 - 1j)
c.Q(0, 1) | ControlledZ(g=1)
c.Q(0) | MeasureHomodyne(phi=pi / 2)
c.Q(1) | MeasureHomodyne(phi=0)

make_figure(c);

You can verify the gates have been added correctly by printing the circuit information:

In [ ]:
print("name        =", c.name)
print("#modes      =", c.n_modes)
print("#operations =", c.n_operations)
print(c)

(sec:circuit-measuring)=

## Measuring a mode

A measurement reads a mode and **consumes it**. The mode is discarded, so nothing can be applied to it
afterwards and it does not appear in the state a simulation reports:

In [ ]:
from claveles.circuit.ops.std import MeasureHomodyne

c = CircuitRepr("measuring")
c.Q(0) | QuantumState.squeezed(r=0.5)
c.Q(1) | QuantumState.vacuum()
c.Q(0) | MeasureHomodyne(phi=0.0)

try:
    c.Q(0) | PhaseRotation(phi=pi)
except RuntimeError as error:
    print("RuntimeError:", error)

Measuring every mode is fine, and leaves a state with no modes at all. That is an ordinary way for a
circuit to end rather than an error, and a simulation reports it as an empty
{py:class}`~claveles.circuit.state.QuantumState`.

A measurement also hands back the value it read, as a
{py:class}`~claveles.circuit.MeasuredVariable`, which is what feedforward is built on — see
{ref}`sec:circuit-adding-feedforward` below.

(sec:circuit-adding-feedforward)=

## Adding feedforward

In Claveles, the parameters of certain operations can depend on the measurement results of other modes.
This mechanism is referred to as **feedforward** in Claveles.

A measurement returns a variable `m` ({py:class}`~claveles.circuit.MeasuredVariable`) that you can
pass straight to an operation, which then follows the value that was read:

```python
c.Q(2) | PhaseRotation(m0)
```

The variable can also be used in an arithmetic expression first. Scaling it, offsetting it, and
combining two results into a complex amplitude all work with the ordinary Python operators:

```python
c.Q(2) | Displacement(m0 - 1j * m1)
```

This covers affine dependencies, which is what the hardware feeds forward. For a non-linear function
of a result, such as its sine, define a feedforward function instead. A feedforward function reads a
single measurement result, so a real parameter, or one part of a complex one, still follows at most
one: a product of two results, or any other combination of them in one quadrature, cannot be fed
forward.

To use a feedforward function:

1. Define a feedforward function `f` ({py:class}`~claveles.feedforward.FeedForwardFunction`)
    * Create a Python function that takes a float as input and returns a float.
    * Decorate the function with {py:deco}`~claveles.feedforward.feedforward`.
    * **Note:** The function must be **self-contained** and able to run independently.
        * Only the `math` module may be used, and it has to be imported **inside** the function.
        * To read a value from the surrounding scope, declare it as a parameter; see
          {ref}`sec:feedforward-parameters`.
2. Obtain the measurement result variable `m` ({py:class}`~claveles.circuit.MeasuredVariable`)
3. Use the feedforwarded parameter `f(m)` as a parameter of an operation.

See [Feedforward](feedforward.ipynb) for the full rules. For example, the teleportation circuit
shown below needs only an affine displacement, so it can be implemented as follows:


In [ ]:
from math import pi

from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import BeamSplitter, Displacement, MeasureHomodyne

# Construct the teleportation circuit.
teleportation = CircuitRepr("teleportation")
teleportation.Q(1, 2) | BeamSplitter(theta=pi / 4, phi=pi)
teleportation.Q(0, 1) | BeamSplitter(theta=pi / 4, phi=pi)
# Measure modes 0 and 1.
m0 = teleportation.Q(0) | MeasureHomodyne(phi=0)
m1 = teleportation.Q(1) | MeasureHomodyne(phi=pi / 2)
# Apply displacement with feedforward.
teleportation.Q(2) | Displacement(m0 - 1j * m1)

make_figure(teleportation);

(sec:circuit-arity)=

## What a circuit takes and leaves behind

Because placements and measurements are part of the sequence, a circuit knows how many modes it expects
an input for and how many it hands back:

* {py:attr}`~claveles.circuit.CircuitRepr.open_modes` — the modes something acts on without the circuit
  placing a state there
* {py:attr}`~claveles.circuit.CircuitRepr.input_arity` — how many of those there are
* {py:attr}`~claveles.circuit.CircuitRepr.output_arity` — how many modes are still carrying a state at the
  end, that is, the ones that were touched and not consumed

In [ ]:
c = CircuitRepr("arity")
c.Q(0) | QuantumState.squeezed(r=0.5)
c.Q(0, 1) | ControlledZ(g=1)  # mode 1 is touched without a state being placed on it
c.Q(0) | MeasureHomodyne(phi=0.0)

print("open_modes   =", c.open_modes)
print("input_arity  =", c.input_arity)
print("output_arity =", c.output_arity)

make_figure(c);

Here mode 1 is open, so the circuit expects one input; mode 0 was measured away, so one mode is left
at the end.

## Editing a circuit

A circuit can be changed after the fact. {py:meth}`~claveles.circuit.CircuitRepr.remove` drops operations
by their position in the sequence, and {py:meth}`~claveles.circuit.CircuitRepr.append` inserts one at a
given position:

In [ ]:
from claveles.circuit.ops.std import Squeezing

c = CircuitRepr("editing")
c.Q(0) | QuantumState.vacuum()
c.Q(0) | PhaseRotation(phi=pi / 2)
c.Q(0) | Squeezing(r=1.0)
print("before      :", c.n_operations, "operations")

c.remove(1)  # drop the phase rotation
print("after remove:", c.n_operations, "operations")

c.append(Displacement(1 + 1j), idx=1, mode=0)
print("after append:", c.n_operations, "operations")

make_figure(c);

Both re-check the whole sequence afterwards, so an edit that would leave the circuit inconsistent — a
state placed on a mode that something already touched, or a measurement removed while a feedforward
still reads it — is refused and the circuit is left as it was.

### Changing a parameter

An operation is **immutable** once built. Its parameters were checked by the constructor that
took them, and {py:meth}`~claveles.circuit.CircuitRepr.get_operation` hands you the circuit's own
operation rather than a copy, so writing to an attribute would skip both that check and the
circuit's record of which operations read which measurements. It raises instead:

In [ ]:
c = CircuitRepr("immutable")
c.Q(0) | QuantumState.vacuum()
c.Q(0) | Squeezing(r=1.0)

op = c.get_operation(1)
try:
    op.r = 1.5
except AttributeError as error:
    print("AttributeError:", error)

To change a parameter, take a copy carrying the new value and put it back where the original was:

* {py:meth}`Operation.replace() <claveles.circuit.ops._base.Operation.replace>`
  takes the arguments the operation's constructor takes, by name, and returns a new operation
* {py:meth}`CircuitRepr.replace_operation() <claveles.circuit.CircuitRepr.replace_operation>`
  puts one at a position in the sequence, leaving it on the modes already at that position

In [ ]:
c.replace_operation(1, op.replace(r=1.5))

print("parameters  =", c.get_operation(1).parameters())
print("unchanged   =", op.parameters())  # the original is left as it was

Because the copy goes through the constructor, it is checked exactly as the original was &mdash; which
is the point of doing it this way rather than by assignment. Which values are refused is up to the
operation; {py:class}`~claveles.circuit.ops.intrinsic.Squeezing45` is one that refuses an angle it cannot
realize:

In [ ]:
from claveles.circuit.ops.intrinsic import Squeezing45

try:
    Squeezing45(theta=0.4).replace(theta=pi)  # a multiple of pi is not a valid angle
except ValueError as error:
    print("ValueError:", error)

{py:meth}`Operation.replace_parameter() <claveles.circuit.ops._base.Operation.replace_parameter>`
takes a parameter's position rather than its name, using the same index as
{py:meth}`Operation.parameters() <claveles.circuit.ops._base.Operation.parameters>` and
{py:meth}`Operation.param_labels() <claveles.circuit.ops._base.Operation.param_labels>`. That is the index a feedforward targets,
and it is what to use to sweep over parameters without knowing what each operation calls them.

{py:meth}`Operation.parameters() <claveles.circuit.ops._base.Operation.parameters>` gives a **tuple**, for the same reason the
operation is immutable: the sequence is built for you rather than held by the operation, so a write
into it would go nowhere. A tuple refuses the write instead of accepting it in silence.

In [ ]:
c = CircuitRepr("sweeping")
c.Q(0) | QuantumState.vacuum()
c.Q(0) | Displacement(1 + 2j)

op = c.get_operation(1)
print("labels =", op.param_labels())
print("values =", op.parameters())

# The real and imaginary parts of `alpha` are two slots of one constructor argument; the half not
# being changed is carried over.
c.replace_operation(1, op.replace_parameter(1, 3.0))
print("after  =", c.get_operation(1).parameters())

```{note}
Replacing a **measurement** with another measurement carries the feedforward over: a
{py:class}`~claveles.circuit.MeasuredVariable` taken from it goes on reading that position, so the
operations driven by it do not have to be rewritten. Replacing one with anything else would leave
them without a source, and is refused.

A feedforward can only read a measurement placed **before** the operation reading it. Since a
measured variable does not exist until its measurement has been placed, building a circuit cannot
break that rule; editing one can, and every edit is checked for it &mdash; as is
{py:meth}`~claveles.circuit.CircuitRepr.proto`, the last point before a circuit leaves the SDK.
```

(sec:circuit-combining)=

## Combining two circuits

`&` lays two circuits side by side, giving one circuit that runs both on separate modes. Neither operand
is modified: the result is an independent circuit holding **copies** of their operations.

The circuit on the left keeps the mode numbers it had, and the one on the right moves up past them.
Numbers are not renumbered from 0, because they are how a circuit's results are read back and they are
allowed to have gaps — closing them would move the results to different modes for no reason:

In [ ]:
sparse = CircuitRepr("sparse")
sparse.Q(0) | QuantumState.squeezed(r=0.5)
sparse.Q(3) | QuantumState.vacuum()  # modes 1 and 2 are not used at all

one_more = CircuitRepr("one_more")
one_more.Q(0) | QuantumState.vacuum()

both = sparse & one_more
print("left       :", sparse.modes)
print("right      :", one_more.modes)
print("composed   :", both.modes)
print("name       :", both.name)
print("operations :", both.n_operations)

make_figure(both);

Mode 3 stays mode 3, the gap at 1 and 2 stays a gap, and the circuit on the right lands on 4 — just above
the highest mode the left one used, rather than filling the gap. Combining *states* follows the same rule;
see {ref}`Working with Quantum States <sec:quantum-states-in-the-sdk>`.

Feedforward is carried over, and it may read *across* the two circuits: an operation in one can read a
measurement in the other, provided the measurement comes first — which for `&` means it is in the circuit
on the left. So a circuit can be split at a measurement and put back together:

In [ ]:
measuring = CircuitRepr("measuring")
measuring.Q(0) | QuantumState.vacuum()
m = measuring.Q(0) | MeasureHomodyne(phi=0.0)

correcting = CircuitRepr("correcting")
correcting.Q(0) | QuantumState.vacuum()
correcting.Q(0) | PhaseRotation(m)  # reads the measurement made in the other circuit

print("feedforwards resolved:", len((measuring & correcting).proto().nlffs))

make_figure(measuring & correcting);

Because the result holds copies, `m` keeps naming the measurement in `measuring` itself rather than the
copy the result contains, so only the `&` that has `measuring` as one of its two operands can resolve it.
Composing `measuring` into something larger first therefore breaks the link, and so does applying an
operation that reads `m` to a circuit that is already composed. Either way serializing says so rather than
dropping the feedforward — keep the measurement and the operation reading it in one circuit, and compose
that circuit directly:

In [ ]:
spare = CircuitRepr("spare")
spare.Q(0) | QuantumState.vacuum()

nested = (measuring & spare) & correcting  # `measuring` is no longer an operand of the outer `&`

try:
    nested.proto()
except ValueError as error:
    print("ValueError:", error)

```{note}
A half on its own cannot be *drawn* either. {py:func}`~claveles.circuit.visualize.make_figure` draws a
feedforwarded parameter as an arrow from the measurement driving it, and in `correcting` alone that
measurement is elsewhere, so there is no arrow to draw: a `ValueError` naming the parameter and the
mode it reads is raised instead. Draw the composed circuit, `measuring & correcting`. The same holds
for halves composed with `@` below.
```

(sec:circuit-feeding)=

### Feeding one circuit into another

`c2 @ c1` runs `c1` first and hands what it leaves to `c2`, the way `f∘g` applies `g` first. A mode
`c1` leaves carrying a state is joined to the mode of `c2` that carries the *same number* and has no
state placed on it, so lining the numbers up is how the connection is expressed.

Nothing is renumbered: every mode of the result keeps the number it had. A number that only one of the
circuits uses is left alone, which the two spare modes below show — `prepare` leaves mode 2 that `use`
never takes, and `use` has an open mode 5 that `prepare` never feeds:

In [ ]:
prepare = CircuitRepr("prepare")
prepare.Q(0) | QuantumState.squeezed(r=0.5)
prepare.Q(2) | QuantumState.vacuum()  # left carrying a state, and never taken

use = CircuitRepr("use")
use.Q(0) | PhaseRotation(phi=pi / 2)  # open: fed by prepare's mode 0
use.Q(5) | Squeezing(r=0.3)  # open: nothing feeds it

print("prepare.output_modes =", prepare.output_modes)
print("use.open_modes       =", use.open_modes)

whole = use @ prepare
print("whole.modes          =", whole.modes)
print("whole.open_modes     =", whole.open_modes)
print("whole.input_arity    =", whole.input_arity)
print("whole.output_modes   =", whole.output_modes)

make_figure(whole);

Mode 0 was the connection, so it is no longer open. Mode 2 stays an output of the result and mode 5 stays
open, both keeping their numbers — nothing was pulled down to fill the gaps at 1, 3 and 4.

A number used by both circuits in any *other* way names two different physical modes, and is refused
rather than quietly renumbered into something you did not write. Composing two circuits that share no
numbers connects nothing at all, which is allowed but warned about.

Feedforward reading from `c1` into `c2` is resolved, which is what makes it possible to split a circuit
at a measurement: the half that measures and the half that corrects can be written and checked
separately, then composed.

The rule is the one `&` follows above: the measurement has to come before the operation reading it, and
both have to be in the two circuits handed to the operator. **The operand that runs first is the opposite
one, though** — `c2 @ c1` runs `c1` — so the measuring half goes on the right here and on the left there.
Getting it backwards raises the same error, and a variable from a circuit composed earlier reaches neither
operand, just as with `&`.

(sec:circuit-saving)=

## Saving and loading

A circuit can be written to a file and read back, which is how a program is kept between
sessions or handed to someone else. The graph and machinery representations have the same pair,
and so do all three result types.

The format is taken from the file name, so a round trip does not have to be told twice which
format it is in:

| Suffix | Format | |
| --- | --- | --- |
| `.json` | JSON | readable, and the easiest to diff |
| `.txt` `.textproto` `.pbtxt` `.prototxt` | protobuf text | readable |
| `.pb` `.bin` | protobuf binary | smallest |
| anything else | {py:data}`~claveles.pb.io.DEFAULT_PROTO_FORMAT` (`text`) | |

`proto_format` decides instead, for a name whose suffix is not in that table or does not say what you
want. Nothing in the file records which format was chosen, so **both ends have to be told**: a file
written with `proto_format` is read back with the same one.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

saved = CircuitRepr("saved")
saved.Q(0) | QuantumState.squeezed(r=0.5)
saved.Q(0) | PhaseRotation(phi=pi / 4)
saved.Q(0) | MeasureHomodyne(phi=0.0)

with TemporaryDirectory() as directory:
    # The suffix decides: `.json` is written and read as JSON with nothing else said.
    by_name = Path(directory) / "circuit.json"
    saved.save(by_name)
    read_back = CircuitRepr.load(by_name)

    # `.dat` is in no row of the table, so without `proto_format` it would be protobuf text.
    overridden = Path(directory) / "circuit.dat"
    saved.save(overridden, proto_format="json")
    print("circuit.dat starts with:", overridden.read_text()[:1])
    also_read_back = CircuitRepr.load(overridden, proto_format="json")

print("name        =", read_back.name)
print("#operations =", read_back.n_operations, "and", also_read_back.n_operations)

A mode left **open** is the one thing a file cannot hold: the format has no way to record that a
mode is waiting for an input, so saving one would silently turn it into a mode carrying the
machine's squeezed state and the circuit read back would differ from the one written.
{py:meth}`~claveles.circuit.CircuitRepr.save` refuses such a circuit unless `default_state` says what
to record in place of the open mode &mdash; `default_state=QuantumState.hardware_squeezed()`, say, which
is what the machine would have supplied. A default stands in for one mode at a time, being placed on each
open mode separately, so it has to be a single-mode state: a wider one names nothing to put on a mode and
is refused.
